In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
//var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D");
var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble3D");

Opening existing database 'P:\hpccluster\RisingBubble3D'.


In [ ]:
var sessions = db.Sessions;
sessions

#0: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_BDF3_testcase1_withOutputs_restart1*	09/10/2024 16:34:05	7ca7d001...
#1: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_singleCore_testcase1*	09/10/2024 12:38:07	cf93bc52...
#2: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_fixedInterfaceCheck_testcase1*	09/05/2024 15:13:05	00b7639e...
#3: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	09/05/2024 08:59:21	d25fc934...
#4: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:41:52	a929ee94...
#5: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1*	08/09/2024 11:54:11	cb26cb13...
#6: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_restart1*	08/05/2024 09:45:57	9d1f82be...
#7: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_setttingsCheck2	08/01/2024 09:27:13	eff0e8c8...
#8: RisingBubble3D	RB3D_16x16x32AMR1_k2_testcase1_setttingsCheck	04/19/2024 16:25:45	19291211...


In [ ]:
var sess = sessions.Pick(1);
sess

RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_singleCore_testcase1*	09/10/2024 12:38:07	cf93bc52...

In [ ]:
sess.Timesteps.TakeLast(10).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble3D__RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_singleCore_testcase1__cf93bc52-4e8f-48f4-b702-5a7f3a0a9c26


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble3D__RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_singleCore_testcase1__cf93bc52-4e8f-48f4-b702-5a7f3a0a9c26

In [ ]:
var tStep_fail = sess.Timesteps.Last();
tStep_fail

 { Time-step: 109; Physical time: 1.0800000000000007s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

In [ ]:
var tStep_OK = sess.Timesteps.Skip(2).First(); //Pick(sess.Timesteps.Count()-2);
tStep_OK

 { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

## Setting up for some checks

In [ ]:
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.LevelSetTools;

In [ ]:
//var tStep = tStep_fail;
var tStep = tStep_OK;

In [ ]:
SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
GridData grdData = (GridData)phi.GridDat; 
LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
levelSet.Acc(1.0, phi); 
LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
// SinglePhaseField phiDG = (SinglePhaseField)tStep.GetField("PhiDG");
// SinglePhaseField phiCP = new SinglePhaseField(new Basis(grdData, phiDG.Basis.Degree + 1), "PhiCP");

// ContinuityProjection contiProj = new ContinuityProjection(
//     phiCP.Basis,
//     phiDG.Basis,
//     grdData,
//     ContinuityProjectionOption.ConstrainedDG);

In [ ]:
// contiProj.MakeContinuous(phiDG, phiCP, LsTrk.Regions.GetCutCellMask(), LsTrk.Regions.GetLevelSetWing(0, +1).VolumeMask);
// levelSet = new LevelSet(phiCP.Basis, "levelSet");
// levelSet.Acc(1.0, phiCP); 
// LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
// LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
int order = 2 * phi.Basis.Degree + 1;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

In [ ]:
SpeciesId spcId_A = LsTrk.GetSpeciesId("A");
SpeciesId spcId_B = LsTrk.GetSpeciesId("B");

## Checking Integral properties

In [ ]:
using BoSSS.Foundation.XDG.Quadrature.HMF;
using BoSSS.Solution.Tecplot;

In [ ]:
//DivergenceFreeBasis DivFreeBasis = new DivergenceFreeBasis(LsTrk.GridDat, LsTrk.GridDat.Cells.RefElements[0], order + 1);
Basis ScalarBasis = new Basis(LsTrk.GridDat, 0); 

In [ ]:
SinglePhaseField testFieldX = new SinglePhaseField(ScalarBasis);
Func<double[], double> fx = (X => 1.0);
testFieldX.ProjectField(fx);
SinglePhaseField testFieldY = new SinglePhaseField(ScalarBasis);
Func<double[], double> fy = (X => 1.0);
testFieldY.ProjectField(fy);
SinglePhaseField testFieldZ = new SinglePhaseField(ScalarBasis);
Func<double[], double> fz = (X => 1.0);
testFieldZ.ProjectField(fz);

In [ ]:
// VectorField<SinglePhaseField> testField = new VectorField<SinglePhaseField>(new SinglePhaseField[] {testFieldX, testFieldY});
VectorField<SinglePhaseField> testField = new VectorField<SinglePhaseField>(new SinglePhaseField[] {testFieldX, testFieldY, testFieldZ});

In [ ]:
CellMask CCmask = LsTrk.Regions.GetCutCellMask();
CCmask.ItemEnum.Count()

80

check Gauss on the whole cutCell domain at one specific timestep

In [ ]:
double gaussOnDomain = 0.0;
CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SchemeHelper.GetLevelSetquadScheme(0, CCmask).Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        int qN = QR.NoOfNodes;
        int D = LsTrk.GridDat.SpatialDimension;

        var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

        for (int d = 0; d < D; d++) {

            var fEval_d = MultidimensionalArray.Create(length, qN);
            testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

            for (int i = 0; i < length; i++) {
                for (int qn = 0; qn < qN; qn++) {
                    EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                }
            }
        }

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            gaussOnDomain += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
gaussOnDomain

9.429561546262649E-08

check Gauss on the whole cutCell domain for a range of timesteps

In [ ]:
var timesteps = sess.Timesteps.TakeLast(10);
timesteps

#0:  { Time-step: 100; Physical time: 1.0000000000000007s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#1:  { Time-step: 101; Physical time: 1.0100000000000007s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#2:  { Time-step: 102; Physical time: 1.0200000000000007s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi,

In [ ]:
bool checkRange = true;

List<double> GaussOnDomainOverTime = new List<double>();
if (checkRange) {
    foreach(var ts in timesteps) {

        SinglePhaseField ts_phi = (SinglePhaseField)ts.GetField("Phi");
        GridData ts_grdData = (GridData)ts_phi.GridDat; 
        LevelSet ts_levelSet = new LevelSet(ts_phi.Basis, "levelSet");
        ts_levelSet.Acc(1.0, ts_phi); 
        LevelSetTracker ts_LsTrk = new LevelSetTracker(ts_grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, ts_levelSet);
        ts_LsTrk.UpdateTracker(ts.PhysicalTime);

        int ts_order = 2 * ts_phi.Basis.Degree + 1;
        XQuadSchemeHelper ts_SchemeHelper = ts_LsTrk.GetXDGSpaceMetrics(ts_LsTrk.SpeciesIdS.ToArray(), ts_order).XQuadSchemeHelper;

        double gaussOnDomain = 0.0;
        CellQuadrature.GetQuadrature(new int[] { 1 }, ts_grdData,
            ts_SchemeHelper.GetLevelSetquadScheme(0, ts_LsTrk.Regions.GetCutCellMask()).Compile(ts_grdData, ts_order),
            delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

                int qN = QR.NoOfNodes;
                int D = ts_grdData.SpatialDimension;

                var LsNormals = ts_LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

                for (int d = 0; d < D; d++) {

                    var fEval_d = MultidimensionalArray.Create(length, qN);
                    testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                    for (int i = 0; i < length; i++) {
                        for (int qn = 0; qn < qN; qn++) {
                            EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                        }
                    }
                }

            },
            delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
                for(int i = 0; i < length; i++) {
                    gaussOnDomain += ResultsOfIntegration[i, 0];
                }
            }
        ).Execute();

        GaussOnDomainOverTime.Add(gaussOnDomain);
    }
}

In [ ]:
GaussOnDomainOverTime

index,value
0,-9.314156956836267E-07
1,3.728202228551851E-05
2,5.868434104503231E-06
3,6.832159905550625E-06
4,5.443306431445784E-05
5,0.00015997704136995125
6,0.00040049700811040437
7,0.0006191589770143027
8,0.0006539690462098735
9,0.0023844010268013385


In [ ]:
public static (double error, double gaussInVolume, double gaussAtInterface, double gaussAtEdges) CheckGaussForCell(int jCell, LevelSetTracker LsTrk, SpeciesId spcId_A, VectorField<SinglePhaseField> testField, int order) {

    GridData grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA); //, MaskType.Geometrical);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    // ========================================================
    double gaussInVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom, order).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            for (int d = 0; d < D; d++) {

                var fGrad_d = MultidimensionalArray.Create(length, qN, D);
                testField[d].EvaluateGradient(i0, length, QR.Nodes, fGrad_d, 0, 0.0);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fGrad_d[i, qn, d];
                    }
                }
            }
            //EvalResult.SetAll(1.0);
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussInVolume += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double gaussAtInterface = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);        // TODO change direction for spcId_B

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                    }
                }
            }
            // EvalResult.SetAll(1.0);
            
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussAtInterface += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);

    double gaussAtCutEdges = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetEdgeQuadScheme(spcId_A, IntegrationDomain: edgeIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            for (int d = 0; d < D; d++) {

                var fEvalEdgeIN_d = MultidimensionalArray.Create(length, qN);
                var fEvalEdgeOUT_d = MultidimensionalArray.Create(length, qN);
                testField[d].EvaluateEdge(i0, length, QR.Nodes, fEvalEdgeIN_d, fEvalEdgeOUT_d);

                for (int i = 0; i < length; i++) {
                    double[] EdgeNormal = LsTrk.GridDat.Edges.NormalsForAffine.ExtractSubArrayShallow(i0 + i, -1).To1DArray();  
                    if (LsTrk.GridDat.Edges.CellIndices[i0 + i, 1] == jCell) {
                        // Console.WriteLine($"jCell {jCell} OUT on edge {i0 + i} - change direction");
                        EdgeNormal.ScaleV(-1.0);
                    }
                    // Console.WriteLine($"Edge normal: ({EdgeNormal[0]}, {EdgeNormal[1]})");

                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEvalEdgeIN_d[i, qn] * EdgeNormal[d];
                    }
                }
            }
            // EvalResult.SetAll(1.0);

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                // Console.WriteLine($"edge {i0 + i}");
                gaussAtCutEdges += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double error = gaussInVolume - gaussAtInterface - gaussAtCutEdges;

    return (error, gaussInVolume, gaussAtInterface, gaussAtCutEdges);

}

In [ ]:
int jCell = LsTrk.Regions.GetCutCellMask().ItemEnum.First();
jCell

181

In [ ]:
CheckGaussForCell(292, LsTrk, spcId_A, testField, order)

Item1,Item2,Item3,Item4
-1.849669733697268E-08,0,-0.016292382852602542,0.01629240134929988


In [ ]:
double totalGaussError = 0.0;
SinglePhaseField gaussErrorField = new SinglePhaseField(new Basis(LsTrk.GridDat, 0), "gaussError");
foreach (int jC in CCmask.ItemEnum) {
    var result = CheckGaussForCell(jC, LsTrk, spcId_A, testField, order);
    Console.WriteLine($"Gauss error in Cell {jC}: {result.error}");
    gaussErrorField.SetMeanValue(jC, result.error);
    totalGaussError += result.error.Abs();
}
totalGaussError

Gauss error in Cell 179: 1.1384122811097797E-18
Gauss error in Cell 180: 5.692061405548898E-19
Gauss error in Cell 195: 5.692061405548898E-19
Gauss error in Cell 196: 0
Gauss error in Cell 290: 2.498902647307677E-11
Gauss error in Cell 291: -2.865111040084578E-08
Gauss error in Cell 292: -1.849669733697268E-08
Gauss error in Cell 293: -7.135945523720211E-11
Gauss error in Cell 306: -2.865111039043744E-08
Gauss error in Cell 307: -9.746841077296065E-10
Gauss error in Cell 308: -1.0408340855860843E-17
Gauss error in Cell 309: 1.849669732222753E-08
Gauss error in Cell 322: -1.8496697340442125E-08
Gauss error in Cell 323: -1.0408340855860843E-17
Gauss error in Cell 324: 9.746841111990534E-10
Gauss error in Cell 325: -1.8496697329166423E-08
Gauss error in Cell 338: 7.135946282661731E-11
Gauss error in Cell 339: 1.8496697319191763E-08
Gauss error in Cell 340: -1.8496697325696976E-08
Gauss error in Cell 341: -7.135944591306342E-11
Gauss error in Cell 403: 1.8634724839594607E-18
Gauss error in

5.373303848799885E-07

In [ ]:
public static (double error, double stokesAtInterface, double stokesAtEdges) CheckStokesForCell(int jCell, LevelSetTracker LsTrk, SpeciesId spcId_A, VectorField<SinglePhaseField> testField, int order) {

    GridData grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA); //, MaskType.Geometrical);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    // ========================================================
    double stokesAtInterface = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom, order).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

            var curv = MultidimensionalArray.Create(length, qN);                              
            ((LevelSet)LsTrk.LevelSets[0]).EvaluateTotalCurvature(i0, length, QR.Nodes, curv);
            // curv.Scale(0.5);    // mean curvature

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {  
                        EvalResult[i, qn, 0] += curv[i, qn] * LsNormals[i, qn, d] * fEval_d[i, qn];
                    }
                }
            }

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                stokesAtInterface += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();
    
    // =========================================================
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);
    
    var testFactory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
    EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(testFactory, edgeIntDom);
    
    double stokesAtEdges = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
    
            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;
    
            for (int i = 0; i < length; i++) {
    
                double[] EdgeNormal = LsTrk.GridDat.Edges.NormalsForAffine.ExtractSubArrayShallow(i0 + i, -1).To1DArray();
                if (LsTrk.GridDat.Edges.CellIndices[i0 + i, 1] == jCell) {
                    // Console.WriteLine($"jCell {jCell} OUT on edge {i0 + i} - change direction");
                    EdgeNormal.ScaleV(-1.0);
                }
                // Console.WriteLine($"normal at Edge {i0+i} : ({EdgeNormal[0]}, {EdgeNormal[1]})");
                // Console.WriteLine($"normal at Edge {i0+i} : ({EdgeNormal[0]}, {EdgeNormal[1]}, , {EdgeNormal[2]})");
                    
                // Console.WriteLine($"edge {i0 + i} - number of nodes {qN}");
                int trf = (LsTrk.GridDat.Edges.CellIndices[i0+i, 0] == jCell) ? LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0 + i, 0] : LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0 + i, 1];
                NodeSet VolNodes = QR.Nodes.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
                // Console.WriteLine($"volume node: ({VolNodes[0, 0]}, {VolNodes[0, 1]})");  
        
                var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(VolNodes, jCell, length);
                // var phiGrad = MultidimensionalArray.Create(length, qN, D);
                // ((LevelSet)LsTrk.LevelSets[0]).EvaluateGradient(jCell, length, VolNodes, phiGrad);
                    
                for (int d = 0; d < D; d++) {
    
                    var fEval = MultidimensionalArray.Create(length, qN);
                    testField[d].Evaluate(jCell, length, VolNodes, fEval);
    
                    for (int qn = 0; qn < qN; qn++) {
                        // Console.WriteLine($"LevelSet normal at Edge {i0+i}: ({LsNormals[i, qn, 0]}, {LsNormals[i, qn, 1]})");
                        // Console.WriteLine($"LevelSet normal at Edge {i0+i}: ({LsNormals[i, qn, 0]}, {LsNormals[i, qn, 1]}, {LsNormals[i, qn, 2]})");
                        // double[] LsNormal = new double[] { phiGrad[i, qn, 0], phiGrad[i, qn, 1] };
                        // LsNormal.Normalize();
                        // Console.WriteLine($"PhiGrad normal at Edge {i0+i}: ({LsNormal[0]}, {LsNormal[1]})");

                        double[] LsTangent = new double[D];
                        for (int d1 = 0; d1 < D; d1++) {
                            for (int d2 = 0; d2 < D; d2++) {
                                double nn = LsNormals[i, qn, d1] * LsNormals[i, qn, d2];
                                if (d1 == d2) {
                                    LsTangent[d1] += (1 - nn) * EdgeNormal[d2];
                                } else {
                                    LsTangent[d1] += -nn * EdgeNormal[d2];
                                }
                            }
                        }         
                        LsTangent = LsTangent.Normalize();

                        // double tangentCheck = GenericBlas.InnerProd(LsTangent, LsNormals.ExtractSubArrayShallow(i, qn, -1).To1DArray());
                        // Console.WriteLine($"LsTangent * LsNormals = {tangentCheck}");                       
    
                        // if (D != 2) 
                        //     throw new ArgumentException();
    
                        // double[] LsTangent = new double[2];
                        
                        // LsTangent[0] = -LsNormals[i, qn, 1];
                        // LsTangent[1] = LsNormals[i, qn, 0];
            
                        // if (GenericBlas.InnerProd(LsTangent, EdgeNormal) < 0.0) {
                        //     // Console.WriteLine($"LsTangent change direction");
                        //     // LsTangent.ScaleV(-1.0);
                        //     Console.WriteLine($"LsTangent NOT pointing outward!");
                        // }
    
                        // Console.WriteLine($"LS tangent: ({LsTangent[0]}, {LsTangent[1]})");  
                        // Console.WriteLine($"LS tangent: ({LsTangent[0]}, {LsTangent[1]} , {LsTangent[2]})");  
                        EvalResult[i, qn, 0] += LsTangent[d] * fEval[i, qn];
                    }
                }
            }
    
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                stokesAtEdges += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double error = stokesAtInterface + stokesAtEdges;

    return (error, stokesAtInterface, stokesAtEdges);
}

In [ ]:
int jCell = LsTrk.Regions.GetCutCellMask().ItemEnum.Last();
jCell

836

In [ ]:
CheckStokesForCell(292, LsTrk, spcId_A, testField, order)

Item1,Item2,Item3
-0.00014292079276251313,-0.1312482828162189,0.13110536202345638


In [ ]:
double totalStokesError = 0.0;
SinglePhaseField stokesErrorField = new SinglePhaseField(new Basis(LsTrk.GridDat, 0), "stokesError");
foreach (int jC in CCmask.ItemEnum) {
    var result = CheckStokesForCell(jC, LsTrk, spcId_A, testField, order);
    Console.WriteLine($"Stokes error in Cell {jC}: {result.error}");
    stokesErrorField.SetMeanValue(jC, result.error);
    totalStokesError += result.error.Abs();
}
totalStokesError

Stokes error in Cell 179: 0
Stokes error in Cell 180: 0
Stokes error in Cell 195: 0
Stokes error in Cell 196: 0
Stokes error in Cell 290: 2.1287227600100267E-05
Stokes error in Cell 291: -5.783878455026259E-05
Stokes error in Cell 292: -0.00014292079276251313
Stokes error in Cell 293: 0.00017604771632516968
Stokes error in Cell 306: -5.783878455020708E-05
Stokes error in Cell 307: -0.0001392783919857976
Stokes error in Cell 308: -0.00010265075869483886
Stokes error in Cell 309: 4.2161646245097995E-06
Stokes error in Cell 322: -0.00014292079276254088
Stokes error in Cell 323: -0.00010265075869486662
Stokes error in Cell 324: -5.3047784722377866E-05
Stokes error in Cell 325: -2.6331114006225564E-05
Stokes error in Cell 338: 0.00014605628077081143
Stokes error in Cell 339: 4.216164624450819E-06
Stokes error in Cell 340: 7.222189394626155E-06
Stokes error in Cell 341: 0.00014651657195460278
Stokes error in Cell 403: 0
Stokes error in Cell 404: 0
Stokes error in Cell 418: -5.783878455020708

0.004013084293093027

In [ ]:
SinglePhaseField totalCurvatureField = new SinglePhaseField(new Basis(LsTrk.GridDat, phi.Basis.Degree - 2), "totalCurvature");
levelSet.ComputeTotalCurvature(totalCurvatureField, CCmask);
// totalCurvatureField.Scale(0.5);  // mean curvature

In [ ]:
DGField[] plotFields = new DGField[] { phi, gaussErrorField, stokesErrorField, totalCurvatureField };
Tecplot("IntegralErrors", tStep.PhysicalTime, 3, plotFields)

Writing output file d:\BoSSS-experimental\public\examples\RisingBubble\IntegralErrors...
done.


In [ ]:
order

11

In [ ]:
jCell = 136;
BitArray cellIntDomBA = new BitArray(LsTrk.GridDat.Cells.NoOfLocalUpdatedCells);
cellIntDomBA[jCell] = true;
CellMask cellIntDom = new CellMask(LsTrk.GridDat, cellIntDomBA); //, MaskType.Geometrical);
var quadScheme = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper.GetLevelSetquadScheme(0, cellIntDom, order);

In [ ]:
quadScheme.FactoryChain.ElementAt(0)

RuleFactory,Domain,Order
{ BoSSS.Foundation.XDG.Quadrature.SayeGaussComboRuleFactory+SayeFactoryWrapper: RefElement: BoSSS.Foundation.Grid.RefElements.Square },<null>,11


In [ ]:
quadScheme.FactoryChain.ElementAt(0).RuleFactory

RefElement
"{ BoSSS.Foundation.Grid.RefElements.Square: Volume: 4, HighestSupportedPolynomialDegree: 23, NoOfFaces: 4, Center: 0 0 , NoOfVertices: 4, FaceRefElement: BoSSS.Foundation.Grid.RefElements.Line, FaceTrafoGramianSqrt: [ 1, 1, 1, 1 ], VertexIndicesToFaces: [ [ 0, 3 ], [ 1, 3 ], [ 0, 2 ], [ 1, 2 ] ], VertexIndicesToCoFaces: [ <null>, <null>, <null>, <null> ], FaceToVertexIndices: [ 0, 2, 1, 3, 2, 3, 0, 1 ], FaceCenters: -1 0 1 0 0 1 0 -1 , FaceNormals: -1 0 1 0 0 1 0 -1 , SpatialDimension: 2, Vertices: -1 -1 1 -1 -1 1 1 1 , HighestKnownOrder: 119, SubdivisonVertices: [ -1 0 -1 -1 0 -1 -1 0 , 0 0 0 -1 1 -1 0 0 , -1 1 -1 0 0 0 -1 1 , 0 1 0 0 1 0 0 1 ], SupportedCellTypes: [ Square_Linear, Square_4, Square_8, Square_9, Square_12, Square_16, Square_25, Square_36, Square_49, Square_64, Square_81, Square_100 ], CoFaceVerticeIndices: [ ], CoFace_FaceIndices: [ ], NoOfCoFaces: 0 }"


In [ ]:
var QR = quadScheme.Compile(LsTrk.GridDat, order).ElementAt(0).Rule;
//QR

In [ ]:
QR.Nodes

Reference,RefElement,NoOfNodes,SpatialDimension,Dimension,Lengths,Storage,IsContinuous,StructureType,Length,NoOfCols,NoOfRows,IsLocked
5574,"{ BoSSS.Foundation.Grid.RefElements.Square: Volume: 4, HighestSupportedPolynomialDegree: 23, NoOfFaces: 4, Center: 0 0 , NoOfVertices: 4, FaceRefElement: BoSSS.Foundation.Grid.RefElements.Line, FaceTrafoGramianSqrt: [ 1, 1, 1, 1 ], VertexIndicesToFaces: [ [ 0, 3 ], [ 1, 3 ], [ 0, 2 ], [ 1, 2 ] ], VertexIndicesToCoFaces: [ <null>, <null>, <null>, <null> ], FaceToVertexIndices: [ 0, 2, 1, 3, 2, 3, 0, 1 ], FaceCenters: -1 0 1 0 0 1 0 -1 , FaceNormals: -1 0 1 0 0 1 0 -1 , SpatialDimension: 2, Vertices: -1 -1 1 -1 -1 1 1 1 , HighestKnownOrder: 119, SubdivisonVertices: [ -1 0 -1 -1 0 -1 -1 0 , 0 0 0 -1 1 -1 0 0 , -1 1 -1 0 0 0 -1 1 , 0 1 0 0 1 0 0 1 ], SupportedCellTypes: [ Square_Linear, Square_4, Square_8, Square_9, Square_12, Square_16, Square_25, Square_36, Square_49, Square_64, Square_81, Square_100 ], CoFaceVerticeIndices: [ ], CoFace_FaceIndices: [ ], NoOfCoFaces: 0 }",6,2,2,"[ 6, 2 ]","[ -0.7514128423518133, 0.4767201476282088, -0.6213497523212244, 0.018376941669623916, -0.4187273185082465, -0.3850133208713721, -0.1899023235376034, -0.6819351593621503, 0.012720110275374397, -0.8721867972745762, 0.14278320030596336, -0.9755054629637269 ]",True,General,12,2,6,True


In [ ]:
QR.Weights.Storage

index,value
0,18.72199185881082
1,18.996143327414767
2,16.792197326936463
3,13.138809488468134
4,9.016817440321137
5,4.130917501799782


## Evaluate Interface Continuity

In [ ]:
int order = phiCP.Basis.Degree;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;
EdgeQuadratureScheme SurfaceElement_Edge = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
//SurfaceElement_Edge.Domain.ItemEnum.Count()

In [ ]:
EdgeMask CutCellBoundaryEdgeMask = LsTrk.Regions.GetCutCellMask().AllEdges().Except(LsTrk.Regions.GetCutCellMask().GetAllInnerEdgesMask());
//CutCellBoundaryEdgeMask.ItemEnum.Count()

In [ ]:
var factory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(factory, CutCellBoundaryEdgeMask);

In [ ]:
double result = 0;
int D = LsTrk.GridDat.SpatialDimension;
EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        // inner quadrature node
        NodeSet Enode_l = QR.Nodes;
        int trf = LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0, 0];
        NodeSet Vnode_l = Enode_l.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
        NodeSet Vnode_g = Vnode_l.CloneAs();
        int cell = LsTrk.GridDat.Edges.CellIndices[i0, 0];
        LsTrk.GridDat.TransformLocal2Global(Vnode_l, Vnode_g, cell);
        if (D == 2)
            Console.WriteLine("inner quadrature node: ({0},{1})", Vnode_g[0, 0], Vnode_g[0, 1]);
        else 
            Console.WriteLine("inner quadrature node: ({0},{1},{2})", Vnode_g[0, 0], Vnode_g[0, 1], Vnode_g[0, 2]);

        // for(int i = 0; i < length; i++) {
        //     EvalResult[i, 0, 0] = 1;    
        // }
        EvalResult.SetAll(1.0);

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            result += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result

0

In [ ]:
CellQuadratureScheme ContactLineVolumeScheme = SchemeHelper.GetContactLineQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
ContactLineVolumeScheme.Domain.ItemEnum.Count()

0